In [ ]:
public class Invoice
{
    public string InvoiceNumber { get; protected set; }
    public DateTime IssueDate { get; protected set; }
    public decimal TotalAmount { get; protected set; }

    //  НОВЫЕ АТРИБУТЫ
    public string Currency { get; set; } = "EUR";
    public decimal Discount { get; set; }
    public bool IsPaid { get; private set; }
    public string CustomerName { get; set; }

    protected List<LineItem> LineItems { get; } = new();

    public Invoice(string invoiceNumber, DateTime issueDate)
    {
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
    }

    //  ПОЛИМОРФИЗМ (override будет ниже)
    public virtual void CalculateTotal()
    {
        TotalAmount = LineItems.Sum(x => x.Total) - Discount;
    }

    //  ПЕРЕГРУЗКА (overload)
    public void CalculateTotal(decimal extraFee)
    {
        CalculateTotal();
        TotalAmount += extraFee;
    }

    public virtual void AddLine(LineItem item)
    {
        LineItems.Add(item);
        CalculateTotal();
    }

    public virtual void RemoveLine(LineItem item)
    {
        LineItems.Remove(item);
        CalculateTotal();
    }

    //  НОВЫЕ МЕТОДЫ
    public void MarkAsPaid()
    {
        IsPaid = true;
    }

    public void ApplyDiscount(decimal value)
    {
        Discount = value;
        CalculateTotal();
    }

    public void PrintInfo()
    {
        Console.WriteLine($"{InvoiceNumber} | {TotalAmount:C} | Paid: {IsPaid}");
    }

    public IReadOnlyList<LineItem> GetLineItems() => LineItems.AsReadOnly();
}

public class GoodsInvoice : Invoice
{
    public DateTime SupplyDate { get; set; }
    public decimal DeliveryCost { get; set; } // новый атрибут

    public GoodsInvoice(string number, DateTime issueDate, DateTime supplyDate)
        : base(number, issueDate)
    {
        SupplyDate = supplyDate;
    }

    //  OVERRIDE
    public override void CalculateTotal()
    {
        base.CalculateTotal();
        TotalAmount += DeliveryCost;
        Console.WriteLine("[GoodsInvoice] Delivery included");
    }

    //  ПЕРЕГРУЗКА
    public void AddLine(LineItem item, bool fragile)
    {
        if (fragile)
            item.UnitPrice += 5;

        base.AddLine(item);
    }
}

public class ServiceInvoice : Invoice
{
    public DateTime ServiceDate { get; set; }
    public decimal HourlyRateBonus { get; set; } // новый атрибут

    public ServiceInvoice(string number, DateTime issueDate, DateTime serviceDate)
        : base(number, issueDate)
    {
        ServiceDate = serviceDate;
    }

    //  OVERRIDE
    public override void CalculateTotal()
    {
        TotalAmount = LineItems.Sum(x => x.Total + HourlyRateBonus);
        TotalAmount -= Discount;

        Console.WriteLine("[ServiceInvoice] Bonus applied");
    }

    // ПЕРЕГРУЗКА
    public void RemoveLine(LineItem item, string reason)
    {
        Console.WriteLine($"Removed: {reason}");
        base.RemoveLine(item);
    }
}

public class CombinedInvoice : Invoice
{
    public bool ReturnAllowed { get; set; }
    public decimal ReturnPenalty { get; set; } // новый атрибут

    public CombinedInvoice(string number, DateTime date, bool returnAllowed)
        : base(number, date)
    {
        ReturnAllowed = returnAllowed;
    }

    //  OVERRIDE
    public override void CalculateTotal()
    {
        decimal total = LineItems.Sum(x => x.Total);
        decimal returned = LineItems.Where(x => x.IsReturned).Sum(x => x.Total);

        TotalAmount = total - returned;

        if (ReturnAllowed)
            TotalAmount -= ReturnPenalty;

        Console.WriteLine("[CombinedInvoice] Return logic applied");
    }
}

public class LineItem
{
    public string Description { get; set; }
    public decimal Quantity { get; set; }
    public decimal UnitPrice { get; set; }
    public bool IsReturned { get; set; }

    public decimal Discount { get; set; } // новый атрибут

    public decimal Total => Quantity * UnitPrice - Discount;

    public LineItem(string desc, decimal qty, decimal price)
    {
        Description = desc;
        Quantity = qty;
        UnitPrice = price;
    }

    //  ПЕРЕГРУЗКА
    public decimal GetTotal(bool includeTax)
    {
        var total = Total;
        return includeTax ? total * 1.2m : total;
    }
}

public class Repository<T>
{
    private readonly List<T> _items = new();

    public void Add(T item) => _items.Add(item);

    public IEnumerable<T> GetAll() => _items;

    public T Find(Func<T, bool> predicate)
    {
        return _items.FirstOrDefault(predicate);
    }
}

public class Processor<T> where T : Invoice
{
    public decimal CalculateTotal(IEnumerable<T> invoices)
    {
        return invoices.Sum(i => i.TotalAmount);
    }

    public void Process(T invoice)
    {
        invoice.CalculateTotal();
        invoice.PrintInfo();
    }
}

class Program
{
    static void Main()
    {
        var goods = new GoodsInvoice("G1", DateTime.Now, DateTime.Now)
        {
            DeliveryCost = 50
        };

        var service = new ServiceInvoice("S1", DateTime.Now, DateTime.Now)
        {
            HourlyRateBonus = 20
        };

        var combined = new CombinedInvoice("C1", DateTime.Now, true)
        {
            ReturnPenalty = 10
        };

        var item = new LineItem("Laptop", 1, 1000);

        goods.AddLine(item);
        service.AddLine(item);
        combined.AddLine(new LineItem("Cable", 2, 10) { IsReturned = true });

        //  Полиморфизм
        List<Invoice> list = new() { goods, service, combined };

        foreach (var inv in list)
        {
            inv.CalculateTotal(); // вызывает РАЗНЫЕ реализации
        }

        //  GENERIC
        var repo = new Repository<Invoice>();
        repo.Add(goods);
        repo.Add(service);

        var processor = new Processor<Invoice>();
        processor.Process(goods);

        Console.WriteLine($"Grand total: {processor.CalculateTotal(list):C}");
    }
}